In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -U ultralytics -q

In [ ]:
import ultralytics
print("Ultralytics version:", ultralytics.__version__)

from ultralytics import YOLO

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26s.pt")

print("YOLO26-S loaded successfully!")

In [ ]:
!find /content/drive/MyDrive/fire_smoke_project -type f -name "*data*yaml" 2>/dev/null

In [ ]:
!cat /content/drive/MyDrive/fire_smoke_project/runs/yolo11s_640_100e_b16_s42/data_colab_used.yaml

In [ ]:
!pip install -q kagglehub

In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download(
    "sayedgamal99/smoke-fire-detection-yolo"
)

print("Dataset downloaded to:")
print(dataset_path)

In [ ]:
import os

for root, dirs, files in os.walk(dataset_path):
    level = root.replace(dataset_path, "").count(os.sep)

    if level <= 3:
        indent = "  " * level
        print(f"{indent}{os.path.basename(root)}/")

In [ ]:
import os

# Create the expected working directory.
base = "/content/datasets/fire_smoke/data"
os.makedirs(base, exist_ok=True)

source_data = os.path.join(dataset_path, "data")

# Create dataset split symlinks.
links = {
    "train": os.path.join(source_data, "training"),
    "val": os.path.join(source_data, "val"),
    "test": os.path.join(source_data, "test"),
}

for name, target in links.items():
    link = os.path.join(base, name)

    if os.path.lexists(link):
        os.remove(link)

    os.symlink(target, link)
    print(f"{link} -> {target}")

print("\nDataset paths prepared.")

In [ ]:
import os
from glob import glob

for split in ["train", "val", "test"]:
    img_path = f"/content/datasets/fire_smoke/data/{split}/images"
    label_path = f"/content/datasets/fire_smoke/data/{split}/labels"

    images = (
        glob(img_path + "/*.jpg") +
        glob(img_path + "/*.jpeg") +
        glob(img_path + "/*.png")
    )

    labels = glob(label_path + "/*.txt")

    print(f"{split}:")
    print(f"  images = {len(images)}")
    print(f"  labels = {len(labels)}")

In [ ]:
from glob import glob

class_ids = set()

for split in ["train", "val", "test"]:
    label_files = glob(
        f"/content/datasets/fire_smoke/data/{split}/labels/*.txt"
    )

    for file in label_files:
        with open(file, "r") as f:
            for line in f:
                if line.strip():
                    class_ids.add(int(line.split()[0]))

print("发现的类别 ID:", sorted(class_ids))

In [ ]:
import os
from glob import glob

print("dataset_path =", dataset_path)

training_path = os.path.join(dataset_path, "data", "training")

print("\ntraining_path =", training_path)
print("training存在吗？", os.path.exists(training_path))

print("\ntraining目录内容：")
if os.path.exists(training_path):
    print(os.listdir(training_path))

print("\ntraining/images 中前10个文件：")
img_dir = os.path.join(training_path, "images")

if os.path.exists(img_dir):
    print(os.listdir(img_dir)[:10])
else:
    print("training/images 不存在")

In [ ]:
!ls -lah /content/datasets/fire_smoke/data/

In [ ]:
import os
from glob import glob

data_root = os.path.join(dataset_path, "data")

# Inspect the downloaded directory names.
print("data 目录里的内容：")
print(os.listdir(data_root))

# Locate the training image directory automatically.
train_source = None

for candidate in ["train", "training"]:
    p = os.path.join(data_root, candidate)
    if os.path.isdir(p) and os.path.isdir(os.path.join(p, "images")):
        train_source = p
        break

print("\n找到的训练集目录：", train_source)

if train_source is None:
    raise RuntimeError("没有找到训练集目录，请把输出截图发给我。")

# Remove an incorrect training symlink if present.
train_link = "/content/datasets/fire_smoke/data/train"

if os.path.islink(train_link):
    os.unlink(train_link)

# Create the corrected symlink.
os.symlink(train_source, train_link)

print("\n新的链接：")
print(train_link, "->", os.readlink(train_link))

# Verify train/validation/test image counts.
print("\n===== 数据集数量检查 =====")

for split in ["train", "val", "test"]:
    img_dir = f"/content/datasets/fire_smoke/data/{split}/images"
    label_dir = f"/content/datasets/fire_smoke/data/{split}/labels"

    images = (
        glob(img_dir + "/*.jpg") +
        glob(img_dir + "/*.jpeg") +
        glob(img_dir + "/*.png") +
        glob(img_dir + "/*.JPG") +
        glob(img_dir + "/*.JPEG") +
        glob(img_dir + "/*.PNG")
    )

    labels = glob(label_dir + "/*.txt")

    print(f"{split}: images = {len(images)}, labels = {len(labels)}")

In [ ]:
yaml_content = """
train: /content/datasets/fire_smoke/data/train/images
val: /content/datasets/fire_smoke/data/val/images
test: /content/datasets/fire_smoke/data/test/images

names:
  0: smoke
  1: fire
"""

yaml_path = "/content/data_yolo26.yaml"

with open(yaml_path, "w") as f:
    f.write(yaml_content)

print("YAML 已创建：", yaml_path)
print()
print(open(yaml_path).read())

In [ ]:
from ultralytics import YOLO

# Start from the official YOLO26-S pretrained weights.
model = YOLO("yolo26s.pt")

results = model.train(
    data="/content/data_yolo26.yaml",

    epochs=100,
    imgsz=640,
    batch=16,
    seed=42,

    device=0,

    project="/content/drive/MyDrive/fire_smoke_project/runs",
    name="yolo26s_640_100e_b16_s42",

    exist_ok=True,
    plots=True,
    save=True,
    verbose=True
)